In [27]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import shutil

# Set device — automatically uses GPU if available, falls back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(torch.cuda.is_available())        # True if NVIDIA GPU detected
print(torch.cuda.get_device_name(0))    # e.g. "NVIDIA GeForce RTX 3080"
print(torch.cuda.memory_allocated(0))   # current memory usage

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'

# Importing the PitStrategyNet model
from src.models.model import PitStrategyNet

# Load data
X_train = np.load(processed_dir / 'X_train.npy')
y_train = np.load(processed_dir / 'y_train.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

Using device: cuda
True
NVIDIA GeForce RTX 3060 Ti
29160448
X_train: (25756, 14), y_train: (25756, 1)


In [28]:
import yaml
from pathlib import Path

project_root = Path.cwd().resolve().parent.parent
with open(project_root / 'src' / 'config.yaml') as f:
    config = yaml.safe_load(f)

model = PitStrategyNet(
    input_dim    = config['model']['input_dim'],
    hidden_dim_1 = config['model']['hidden_dim_1'],
    hidden_dim_2 = config['model']['hidden_dim_2'],
    dropout_rate = config['model']['dropout_rate']
).to(device)

pit_count     = (y_train == 1).sum()
non_pit_count = (y_train == 0).sum()
pos_weight    = torch.tensor([non_pit_count / pit_count], dtype=torch.float32)
criterion     = nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)

optimizer  = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
num_epochs = config['training']['epochs']
batch_size = config['training']['batch_size']

In [29]:
class LapDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(LapDataset(X_train, y_train), batch_size=batch_size, shuffle=False, pin_memory=False, num_workers=0)

In [30]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.01

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping to prevent loss going to infinity
        optimizer.step()
        train_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {train_loss/len(train_loader):.4f}")

# Save weights
models_dir = project_root / 'src' / 'models' / config['paths']['folder_name']
models_dir.mkdir(exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': config['model']['input_dim'],
}, models_dir / config['paths']['model_filename'])

shutil.copy(project_root / 'src' / 'config.yaml',
            models_dir / config['paths']['config_name'])

print("Model saved.")

Epoch 10/250 | Train Loss: 0.6063
Epoch 20/250 | Train Loss: 0.5320
Epoch 30/250 | Train Loss: 0.4856
Epoch 40/250 | Train Loss: 0.4415
Epoch 50/250 | Train Loss: 0.4082
Epoch 60/250 | Train Loss: 0.3798
Epoch 70/250 | Train Loss: 0.3580
Epoch 80/250 | Train Loss: 0.3380
Epoch 90/250 | Train Loss: 0.3197
Epoch 100/250 | Train Loss: 0.3097
Epoch 110/250 | Train Loss: 0.2916
Epoch 120/250 | Train Loss: 0.2790
Epoch 130/250 | Train Loss: 0.2658
Epoch 140/250 | Train Loss: 0.2615
Epoch 150/250 | Train Loss: 0.2487
Epoch 160/250 | Train Loss: 0.2360
Epoch 170/250 | Train Loss: 0.2252
Epoch 180/250 | Train Loss: 0.2228
Epoch 190/250 | Train Loss: 0.2124
Epoch 200/250 | Train Loss: 0.2055
Epoch 210/250 | Train Loss: 0.1979
Epoch 220/250 | Train Loss: 0.1938
Epoch 230/250 | Train Loss: 0.1895
Epoch 240/250 | Train Loss: 0.1785
Epoch 250/250 | Train Loss: 0.1763
Model saved.
